# 2021 municipal election results: cleaning

Goal: turn the IEC's raw 2021 results into one clean table with one row per municipality.

Input: the provincial CSV files in `MyDrive/DIRISA_SDC/data/raw/iec_lge/2021/`
Output: `MyDrive/DIRISA_SDC/data/interim/results_municipal_2021_clean.csv`

Steps:
1. Load the provincial files and put them in one table
2. Keep the ward ballot only
3. Keep one row per voting station
4. Add up to municipality level
5. Work out turnout
6. Check the results
7. Save the clean file

The 2011, 2016 and 2021 notebooks are the same. Only `YEAR` in the setup cell changes.

## Setup

In [ ]:
import re
from pathlib import Path

import pandas as pd
from google.colab import drive

# Connect Google Drive so the notebook can read the raw files
drive.mount('/content/drive')

# ---- Settings (the only thing that differs between the notebooks) ----
YEAR = 2021

# ---- Paths ----
DRIVE   = Path("/content/drive/MyDrive")
RAW     = DRIVE / "DIRISA_SDC" / "data" / "raw" / "iec_lge" / str(YEAR)   # raw provincial CSV files
INTERIM = DRIVE / "DIRISA_SDC" / "data" / "interim"   # where clean files are saved
INTERIM.mkdir(parents=True, exist_ok=True)

print(f"Cleaning year: {YEAR}")
print("Raw folder exists:", RAW.exists())

## Step 1: Load and combine the provincial files

The IEC publishes one CSV per province. Some files are saved in different text encodings, so the reader below tries several encodings until the header reads properly.

In [ ]:
def read_csv_any_encoding(path):
    """Read a CSV, trying common encodings until the header looks correct."""
    for enc in ("utf-8", "utf-8-sig", "utf-16", "latin-1"):
        try:
            df = pd.read_csv(path, encoding=enc)
            # A correctly read file has several columns, and its first column
            # name does not start with "ÿ" (a sign the encoding was wrong).
            if df.shape[1] > 1 and not str(df.columns[0]).startswith("ÿ"):
                return df
        except (UnicodeDecodeError, UnicodeError):
            continue
    # Last resort: accept whatever latin-1 gives us
    return pd.read_csv(path, encoding="latin-1")


province_files = sorted(RAW.glob("*.csv"))
print(f"{len(province_files)} province file(s) found:")
for f in province_files:
    print(" -", f.name)

# Files such as "KN (1).csv" are accidental re-downloads. Left in the folder,
# they would be counted twice and inflate every total.
duplicates = [f.name for f in province_files if re.search(r" \(\d+\)\.csv$", f.name)]
if duplicates:
    print("\nWARNING: possible duplicate downloads, remove them from the folder:", duplicates)

# Stack all provinces into one table, remembering which file each row came from
raw = pd.concat(
    [read_csv_any_encoding(f).assign(source_file=f.name) for f in province_files],
    ignore_index=True,
    sort=False,
)

print(f"\nCombined shape: {raw.shape}")
print("Columns:", list(raw.columns))
raw.head()

### Check: are these files really from this election?

Every row in an IEC file has `DateGenerated`, the date the file was made. If files from another election end up
in this folder by mistake, everything after this is wrong. This happened once: the 2016 folder had the 2021 files
in it.

Some files were made after election day, so they show the next year (some 2016 files say 2017). We accept `YEAR`
and `YEAR + 1`. We also expect exactly nine provinces.

In [ ]:
# Years that appear in DateGenerated, per file
file_years = (
    raw.groupby("source_file")["DateGenerated"]
    .agg(lambda s: sorted(set(s.astype(str).str.extract(r"(\d{4})")[0].dropna().astype(int))))
)
print(file_years.to_string())

wrong_year = {f: yrs for f, yrs in file_years.items() if not set(yrs) <= {YEAR, YEAR + 1}}
assert not wrong_year, (
    f"These files in {RAW} are not from the {YEAR} election: {wrong_year}. "
    f"Replace them with the {YEAR} downloads before continuing."
)

n_provinces = raw["Province"].nunique()
assert n_provinces == 9, f"Expected 9 provinces, found {n_provinces}: {sorted(raw['Province'].unique())}"
print(f"\nOK: all files are from the {YEAR} election, 9 provinces.")

## Step 2: Keep the ward ballot only

Each voting station reports more than one ballot (for example the ward ballot and the party list ballot). Using both would count the same voters twice, so we keep only the ward ballot.

In [ ]:
print("Ballot types in the raw data:")
print(raw["BallotType"].value_counts())

ward = raw[raw["BallotType"] == "Ward"].copy()
print(f"\nRows after keeping Ward only: {len(ward)} (was {len(raw)})")

## Step 3: One row per voting station

The file has one row for each party at each station, so `RegisteredVoters` and `SpoiltVotes` repeat on every
party row. Adding them up directly would count them many times. Keeping one row per station fixes this.

We use `VotingDistrict` (a unique number) to identify stations, not the station name, because some
municipalities have two stations with the same name.

`TotalValidVotes` is different: it is a count per party, so in Step 4 we add it up over all party rows.

In [ ]:
one_row_per_station = ward.drop_duplicates(
    subset=["Province", "Municipality", "VotingDistrict"]
)
print(f"Voting stations (Ward ballot): {len(one_row_per_station)}")

## Step 4: Aggregate to municipality level

In [ ]:
# Registered voters and spoilt votes: sum over stations (one row each)
registered_and_spoilt = (
    one_row_per_station
    .groupby(["Province", "Municipality"])[["RegisteredVoters", "SpoiltVotes"]]
    .sum()
    .reset_index()
)

# Valid votes: sum over every party row of the Ward ballot
valid_votes = (
    ward
    .groupby(["Province", "Municipality"])["TotalValidVotes"]
    .sum()
    .reset_index()
)

# Join the two tables (a municipality must appear in both to be kept)
muni = registered_and_spoilt.merge(valid_votes, on=["Province", "Municipality"], how="inner")
print(f"Municipalities: {len(muni)}")

## Step 5: Turnout

Turnout = (valid votes + spoilt votes) / registered voters x 100

In [ ]:
muni["total_votes_cast"] = muni["TotalValidVotes"] + muni["SpoiltVotes"]
muni["turnout_pct"] = muni["total_votes_cast"] / muni["RegisteredVoters"] * 100
muni["year"] = YEAR

# Fixed column order so the 2011, 2016 and 2021 files line up
muni = muni[[
    "year", "Province", "Municipality",
    "RegisteredVoters", "TotalValidVotes", "SpoiltVotes",
    "total_votes_cast", "turnout_pct",
]]

muni.head(10)

## Step 6: Checks

Turnout should be between 0% and 100%, and the national figure should look like a normal South African local
election.

In [ ]:
# 1. Spread of municipal turnout
print(muni["turnout_pct"].describe())

# 2. National turnout (uses the municipal totals, so it ties back to the table)
national_turnout = muni["total_votes_cast"].sum() / muni["RegisteredVoters"].sum() * 100
print(f"\nNational turnout check: {national_turnout:.1f}%")

# 3. Structural checks
print("Rows with turnout outside 0-100%:", ((muni["turnout_pct"] < 0) | (muni["turnout_pct"] > 100)).sum())
print("Missing values:", int(muni.isna().sum().sum()))
assert not muni.duplicated(["Province", "Municipality"]).any(), "Duplicate municipalities found"

## Step 7: Save the clean file

In [ ]:
out_path = INTERIM / f"results_municipal_{YEAR}_clean.csv"
muni.to_csv(out_path, index=False)
print(f"Saved to: {out_path}")